# 03 — Fine-tune Qwen2.5-7B with QLoRA on Kaggle

Runs on Kaggle (P100 16GB or T4 ×2 30GB, 12-hour session, 30 GPU-hr/week).

**Setup before running**:
1. Create a new Kaggle notebook, **Settings → Accelerator → GPU P100** (or T4 ×2).
2. **Settings → Internet → On** (needed for `pip install` and HF Hub).
3. **Add-ons → Secrets → add `HF_TOKEN`** with write access (for pushing the adapter to `Tamir39/...`).
4. **Run all cells**. Total wall time ≈ 1.5–2 h for 3 epochs over ~305 examples.

**Pipeline**:
1. Clone the repo into `/kaggle/working`.
2. `pip install` peft / trl / bitsandbytes / sentence-transformers.
3. Authenticate with HF using the Kaggle secret.
4. Build the SFT dataset from `data/qa/train_qa.jsonl` (mixed-context mode).
5. QLoRA-fine-tune Qwen2.5-7B-Instruct (4-bit nf4) for 3 epochs.
6. Save adapter under `checkpoints/` and push to `Tamir39/qwen2_5-7b-vietnam-tax-lora`.

In [ ]:
# 1. Clone the repo into /kaggle/working (replace REPO_URL if you fork it).
import os, subprocess

REPO_URL = "https://github.com/tamir39/rag-llm-vietnam-law-advisor.git"
REPO_DIR = "/kaggle/working/LawMate"

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", "develop", REPO_URL, REPO_DIR])

os.chdir(REPO_DIR)
print(subprocess.check_output(["git", "log", "-1", "--oneline"]).decode().strip())

In [ ]:
# 2. Install pinned dependencies (Kaggle base already has torch/transformers/datasets;
#    we just upgrade the QLoRA stack).
%pip install -q -U "peft>=0.12" "trl>=0.9" "bitsandbytes>=0.43" "accelerate>=0.33" "sentence-transformers>=3.0"

In [ ]:
# 3. Authenticate with the HF Hub via the Kaggle secret.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN, add_to_git_credential=False)
print("HF login OK")

In [ ]:
# 4. Build the SFT dataset and inspect a sample.
import sys
sys.path.insert(0, REPO_DIR)

from src.config import BASE_LLM, TRAIN_QA, KB_CSV
from src.finetune.dataset import build_sft_dataset
from src.finetune.trainer import load_tokenizer

tokenizer = load_tokenizer(BASE_LLM)
train_ds = build_sft_dataset(TRAIN_QA, tokenizer, kb_path=KB_CSV, context_mode="mixed")
print(f"train: {len(train_ds)} examples")
print("---- sample ----")
print(train_ds[0]["text"][:800])

In [ ]:
# 5. Run QLoRA fine-tuning. ~1.5–2 h on P100 for 3 epochs.
from src.config import LORA_ADAPTER
from src.finetune.trainer import train

adapter_dir = train(
    base_model_id=BASE_LLM,
    dataset=train_ds,
    output_dir=LORA_ADAPTER,
)
print(f"Adapter saved to {adapter_dir}")

In [ ]:
# 6. Push the adapter to the HF Hub.
from huggingface_hub import HfApi, create_repo

HF_REPO_ID = "Tamir39/qwen2_5-7b-vietnam-tax-lora"

create_repo(HF_REPO_ID, exist_ok=True, private=False)
HfApi().upload_folder(
    folder_path=str(adapter_dir),
    repo_id=HF_REPO_ID,
    commit_message="upload QLoRA adapter (3 epochs, mixed-context, 5 VN tax laws)",
)
print(f"Pushed → https://huggingface.co/{HF_REPO_ID}")